# 04 最终九类因子回测

回测函数集中在 `src/treasury_futures/factors/cicc_close_session_reverse/backtest.py`；本 Notebook 只保留筛选配置、运行和结果展示。


In [ ]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from IPython.display import display
from treasury_futures.factors.cicc_close_session_reverse.backtest import *  # noqa: F403


In [ ]:

# 回测信号筛选配置：None 表示该维度不过滤。
# factor_ids 示例：frozenset({"closing_capital_oi_accel_h3"})
# categories 示例：frozenset({"2. 尾盘资金进场"})
# roll_statuses 示例：frozenset({"CROSSOVER","ACTIVE", "WATCH","NORMAL"})
signal_filter = SignalFilterConfig(
    exclude_blocked=True,       # 默认排除 block_signal=True 的移仓/数据异常信号
    factor_ids=None,            # 只回测指定 factor_id；None=全部
    categories=None,            # 只回测指定类别；None=全部
    roll_statuses={"ACTIVE","WATCH","NORMAL"},         # 只回测指定移仓状态；None=全部未被 block 的状态
    signal_start_date=None,     # 例如 "2022-01-01"
    signal_end_date=None,       # 例如 "2025-12-31"
)

result = run_backtest(signal_filter=signal_filter)
print(f'backtest workbook: {FACTOR_OUTPUT_DIR / BACKTEST_WORKBOOK}')


## 完整逐笔交易
逐笔展示每个语义因子的信号日、入场日、出场日、价格与扣费后 PnL。

In [ ]:
display(result.trades)

## 终端日历不足信号审计
只允许样本末端无法取得 T+持有期出场日的信号进入本表；其他日历或分钟窗口问题仍直接报错。

In [ ]:
display(result.dropped_signals)

## 因子统计
按中文因子名称汇总胜率、均值与累计收益。

In [ ]:
display(result.factor_stats)

## 类别统计、总体统计、年度统计
总体统计固定展示输入信号数、可执行交易数、终端丢弃数；类别与年度口径用于导师快速复核组合贡献。

In [ ]:
display(result.category_stats)
display(result.overall)
display(result.yearly)

## 综合回测图
价格和九个因子信号、累积净 PnL、回撤使用同一时间轴联动展示。颜色表示四大类别，形状区分信号组。

In [ ]:
result.combined_figure.show(config={'responsive': True, 'displaylogo': False})

## 持仓时间与策略年化绩效

持仓比例使用“至少有一笔仓位存续的交易日数 / 首次入场至最后退出之间的总交易日数”，重叠交易只计一次；同时报告累计单笔持仓天数，以观察重叠仓位。

默认只统计 `POST_OPEN_SWITCH = 2020-07-20` 之后的信号，与综合回测图的展示区间一致。收益口径与本 Notebook 现有组合曲线保持一致：每笔交易的 `pnl_bp_net` 在退出日确认，同日多笔交易直接相加，非退出日收益记为 0；按 252 个交易日年化，无风险利率取 0。该口径不是逐日盯市，也没有对重叠仓位进行资金归一化。

In [ ]:
TRADING_DAYS_PER_YEAR = 252
PERFORMANCE_START_DATE = POST_OPEN_SWITCH  # 与综合回测图一致；设为 None 可统计全部历史

trades_for_metrics = result.trades.copy()
trades_for_metrics['signal_date'] = pd.to_datetime(trades_for_metrics['signal_date']).dt.normalize()
if PERFORMANCE_START_DATE is not None:
    performance_start_date = pd.Timestamp(PERFORMANCE_START_DATE).normalize()
    trades_for_metrics = trades_for_metrics[trades_for_metrics['signal_date'] >= performance_start_date].copy()
if trades_for_metrics.empty:
    raise BacktestInputError('没有可执行交易，无法计算持仓比例和年化绩效。')

# 统一日期格式，并以第一笔入场到最后一笔退出作为策略实际回测区间。
trades_for_metrics['entry_date'] = pd.to_datetime(trades_for_metrics['entry_date']).dt.normalize()
trades_for_metrics['exit_date'] = pd.to_datetime(trades_for_metrics['exit_date']).dt.normalize()
first_entry_date = trades_for_metrics['entry_date'].min()
last_exit_date = trades_for_metrics['exit_date'].max()

# 使用本次回测撮合时已经生成的真实交易日历，不重复读取 Parquet。
daily_calendar = pd.Series(pd.to_datetime(result.trading_calendar)).dt.normalize()
backtest_calendar = pd.DatetimeIndex(
    daily_calendar[(daily_calendar >= first_entry_date) & (daily_calendar <= last_exit_date)]
    .drop_duplicates()
    .sort_values()
)
if backtest_calendar.empty:
    raise BacktestInputError('首次入场和最后退出之间没有可用交易日历。')

# 去重后的有仓交易日：任意一笔交易满足 entry_date <= 当日 <= exit_date 即认为当日有仓。
held_trading_mask = pd.Series(False, index=backtest_calendar)
for trade in trades_for_metrics.itertuples(index=False):
    held_trading_mask |= (backtest_calendar >= trade.entry_date) & (backtest_calendar <= trade.exit_date)

total_trading_days = int(len(backtest_calendar))
held_trading_days = int(held_trading_mask.sum())
trading_day_exposure = held_trading_days / total_trading_days

# 累计单笔持仓天数会重复计算重叠仓位，因此可能高于总交易日数。
summed_position_days = int(trades_for_metrics['hold_days'].sum())
gross_position_day_ratio = summed_position_days / total_trading_days

# 自然日口径包含隔夜、周末和节假日，同样对重叠仓位去重。
calendar_dates = pd.date_range(first_entry_date, last_exit_date, freq='D')
held_calendar_mask = pd.Series(False, index=calendar_dates)
for trade in trades_for_metrics.itertuples(index=False):
    held_calendar_mask |= (calendar_dates >= trade.entry_date) & (calendar_dates <= trade.exit_date)
total_calendar_days = int(len(calendar_dates))
held_calendar_days = int(held_calendar_mask.sum())
calendar_day_exposure = held_calendar_days / total_calendar_days

# 沿用当前回测口径：在 exit_date 汇总已经扣除交易成本的净 PnL，并补齐无退出交易的 0 收益日。
exit_day_pnl_bp = trades_for_metrics.groupby('exit_date')['pnl_bp_net'].sum()
daily_strategy_returns = pd.DataFrame({'trade_date': backtest_calendar})
daily_strategy_returns['pnl_bp_net'] = exit_day_pnl_bp.reindex(backtest_calendar, fill_value=0.0).to_numpy()
daily_strategy_returns['daily_return'] = daily_strategy_returns['pnl_bp_net'] / 10_000.0
if (daily_strategy_returns['daily_return'] <= -1).any():
    raise BacktestInputError('存在单日收益率 <= -100%，无法计算复合收益。')

daily_strategy_returns['equity_curve'] = (1.0 + daily_strategy_returns['daily_return']).cumprod()
daily_strategy_returns['running_peak'] = daily_strategy_returns['equity_curve'].cummax().clip(lower=1.0)
daily_strategy_returns['drawdown'] = daily_strategy_returns['equity_curve'] / daily_strategy_returns['running_peak'] - 1.0

daily_return = daily_strategy_returns['daily_return']
total_return = float(daily_strategy_returns['equity_curve'].iloc[-1] - 1.0)
annual_return = float((1.0 + total_return) ** (TRADING_DAYS_PER_YEAR / total_trading_days) - 1.0)
daily_volatility = float(daily_return.std(ddof=1)) if len(daily_return) > 1 else np.nan
annual_volatility = daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)
sharpe = (float(daily_return.mean()) / daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)) if daily_volatility > 0 else np.nan
max_drawdown = float(daily_strategy_returns['drawdown'].min())
calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

# 保留原始数值，方便后续程序化使用。
strategy_metrics_raw = {
    'first_entry_date': first_entry_date,
    'last_exit_date': last_exit_date,
    'total_calendar_days': total_calendar_days,
    'total_trading_days': total_trading_days,
    'signal_day_count': int(trades_for_metrics['signal_date'].nunique()),
    'trade_count': len(trades_for_metrics),
    'summed_position_days': summed_position_days,
    'held_trading_days': held_trading_days,
    'trading_day_exposure': trading_day_exposure,
    'gross_position_day_ratio': gross_position_day_ratio,
    'held_calendar_days': held_calendar_days,
    'calendar_day_exposure': calendar_day_exposure,
    'total_net_pnl_bp': float(daily_strategy_returns['pnl_bp_net'].sum()),
    'total_return': total_return,
    'annual_return': annual_return,
    'annual_volatility': annual_volatility,
    'sharpe': sharpe,
    'max_drawdown': max_drawdown,
    'calmar': calmar,
}

strategy_metrics = pd.DataFrame([
    {'指标': '首次入场日期', '数值': first_entry_date.strftime('%Y-%m-%d'), '说明': '策略实际开始持仓'},
    {'指标': '最后退出日期', '数值': last_exit_date.strftime('%Y-%m-%d'), '说明': '策略最后一笔交易退出'},
    {'指标': '总自然日时长', '数值': f'{total_calendar_days:,} 天', '说明': '首次入场至最后退出，首尾均包含'},
    {'指标': '总交易日时长', '数值': f'{total_trading_days:,} 天', '说明': '同一区间内的真实交易日数量'},
    {'指标': '纳入信号日数', '数值': f"{trades_for_metrics['signal_date'].nunique():,} 天", '说明': '去重后的 signal_date 数量'},
    {'指标': '完成交易数', '数值': f'{len(trades_for_metrics):,} 笔', '说明': '一个信号日可由多个因子产生多笔交易'},
    {'指标': '累计单笔持仓天数', '数值': f'{summed_position_days:,} 仓位日', '说明': '各笔 hold_days 相加，重叠仓位重复计算'},
    {'指标': '实际有仓交易日数', '数值': f'{held_trading_days:,} 天', '说明': '至少有一笔仓位，重叠仓位只计一次'},
    {'指标': '交易日持仓比例', '数值': f'{trading_day_exposure:.2%}', '说明': '实际有仓交易日数 / 总交易日时长'},
    {'指标': '累计仓位日比例', '数值': f'{gross_position_day_ratio:.2%}', '说明': '累计单笔持仓天数 / 总交易日；可能超过100%'},
    {'指标': '实际有仓自然日数', '数值': f'{held_calendar_days:,} 天', '说明': '包括隔夜、周末和节假日，重叠去重'},
    {'指标': '自然日持仓比例', '数值': f'{calendar_day_exposure:.2%}', '说明': '实际有仓自然日数 / 总自然日时长'},
    {'指标': '总净 PnL', '数值': f"{strategy_metrics_raw['total_net_pnl_bp']:+,.2f} bp", '说明': '所有交易 pnl_bp_net 之和'}
])

display(strategy_metrics)
